In [1]:
import pandas as pd
from imblearn.under_sampling import RandomUnderSampler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from collections import Counter

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## PPP

In [2]:
random_state = 42

In [3]:
classification_reports = []
classification_report_keys = []

### Data

In [4]:
# Import credit card datasaet
df = pd.read_csv('C:/Users/caleb/Projects/BU Spring 2026/Module-B-semester-2/Milestone 3 EDA/ppp_cleaned.csv')
df

,ProcessingMethod,BorrowerState,LoanStatus,Term,InitialApprovalAmount,CurrentApprovalAmount,ServicingLenderState,RuralUrbanIndicator,HubzoneIndicator,LMIIndicator,...,MORTGAGE_INTEREST_PROCEED_pct,RENT_PROCEED_pct,REFINANCE_EIDL_PROCEED_pct,HEALTH_CARE_PROCEED_pct,DEBT_INTEREST_PROCEED_pct,PROCEED_Per_Job,Fraud,DateApproved_int,ForgivenessDate_int,LoanStatusDate_int
0,0.0,48.0,2.0,24,769358.78,769358.78,11.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,12409.01,0,1588291200,1605830400,1608249600
1,0.0,48.0,2.0,24,736927.79,736927.79,11.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,10094.90,0,1588291200,1628726400,1632787200
2,0.0,48.0,2.0,24,691355.00,691355.00,29.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,9218.07,0,1588291200,1612915200,1615939200
3,0.0,48.0,2.0,24,499871.00,499871.00,29.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,23803.38,0,1588291200,1631232000,1634342400
4,0.0,48.0,2.0,24,367437.00,367437.00,37.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,14697.48,0,1588291200,1617840000,1629158400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
968527,0.0,56.0,2.0,24,150000.00,150000.00,54.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,10000.00,0,1585872000,1607472000,1610496000
968528,0.0,56.0,2.0,24,150000.00,150000.00,31.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,3452.38,0,1586822400,1604361600,1607385600
968529,1.0,56.0,2.0,60,150000.00,150000.00,54.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,29999.40,0,1613088000,1629158400,1631664000
968530,0.0,56.0,2.0,60,150000.00,150000.00,18.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,21428.57,0,1586908800,1645574400,1646697600


### Undersampling

In [5]:
X = df.drop(columns='Fraud')
y = df['Fraud']
print("Original dataset shape:", Counter(y))
rus = RandomUnderSampler(sampling_strategy=0.1, random_state=42)
X_resampled, y_resampled = rus.fit_resample(X, y)

print("Resampled dataset shape:", Counter(y_resampled))

Original dataset shape: Counter({0: 968437, 1: 95})
Resampled dataset shape: Counter({0: 950, 1: 95})


### Split Data

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, 
                                                    y_resampled, 
                                                    test_size=0.2, 
                                                    stratify=y_resampled,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

### Baseline Models
#### (No parameter optimization, feature scaling, or cross validation)

#### Linear Regression

In [7]:
model = LogisticRegression()

In [8]:
model.fit(X_train, y_train)

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [9]:
y_pred = model.predict(X_test)

In [10]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline Logistic Regression')

              precision    recall  f1-score   support

           0       0.99      0.89      0.94       190
           1       0.45      0.89      0.60        19

    accuracy                           0.89       209
   macro avg       0.72      0.89      0.77       209
weighted avg       0.94      0.89      0.91       209



#### Ridge

In [11]:
from sklearn.linear_model import RidgeClassifier


ridge_model = RidgeClassifier(random_state=random_state)

In [12]:
ridge_model.fit(X_train, y_train)

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=3.15646e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,class_weight,None
,solver,'auto'
,positive,False
,random_state,42


In [13]:
y_pred = ridge_model.predict(X_test)

In [14]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline Ridge Classifier')


              precision    recall  f1-score   support

           0       0.98      0.95      0.97       190
           1       0.62      0.79      0.70        19

    accuracy                           0.94       209
   macro avg       0.80      0.87      0.83       209
weighted avg       0.95      0.94      0.94       209



#### Lasso

In [15]:
lasso_model = LogisticRegression(penalty='l1', solver='liblinear')

In [16]:
lasso_model.fit(X_train, y_train)

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,penalty,'l1'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [17]:
y_pred = lasso_model.predict(X_test)

In [18]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline L1 (Lasso) Logistic Regression')

              precision    recall  f1-score   support

           0       0.97      0.97      0.97       190
           1       0.70      0.74      0.72        19

    accuracy                           0.95       209
   macro avg       0.84      0.85      0.84       209
weighted avg       0.95      0.95      0.95       209



### Scaling Data

In [19]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Parameter Tuning

#### Logistic Regression

In [20]:
# We can only tune the C (float) and Class_weight to tryout balanced. 

In [21]:
# cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)
from sklearn.model_selection import StratifiedKFold


cv = StratifiedKFold(n_splits=5)

In [22]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])
    
    model = LogisticRegression(
        class_weight=class_weight,
        C=c,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

log_reg_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_log_reg_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_log_reg_unscaled_recall_optuna_results)

[I 2026-06-24 19:23:16,086] A new study created in memory with name: no-name-f2d90437-e943-4eb1-8177-11a61af9fed8
[I 2026-06-24 19:23:17,475] Trial 0 finished with value: 0.79 and parameters: {'C': 0.8618697353185664, 'Class_weight': None}. Best is trial 0 with value: 0.79.
[I 2026-06-24 19:23:18,648] Trial 1 finished with value: 0.9225000000000001 and parameters: {'C': 0.8297147054295477, 'Class_weight': 'balanced'}. Best is trial 1 with value: 0.9225000000000001.
[I 2026-06-24 19:23:19,663] Trial 2 finished with value: 0.79 and parameters: {'C': 0.599220617113706, 'Class_weight': None}. Best is trial 1 with value: 0.9225000000000001.
[I 2026-06-24 19:23:20,717] Trial 3 finished with value: 0.7766666666666667 and parameters: {'C': 0.937685201065128, 'Class_weight': None}. Best is trial 1 with value: 0.9225000000000001.
[I 2026-06-24 19:23:20,733] Trial 4 finished with value: 0.9225000000000001 and parameters: {'C': 0.10625805127345389, 'Class_weight': 'balanced'}. Best is trial 1 with

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,state
49,49,0.935,2026-06-24 19:23:21.467203,2026-06-24 19:23:21.482290,0 days 00:00:00.015087,0.002577,balanced,COMPLETE
51,51,0.935,2026-06-24 19:23:21.499414,2026-06-24 19:23:21.516371,0 days 00:00:00.016957,0.027829,balanced,COMPLETE
41,41,0.935,2026-06-24 19:23:21.340383,2026-06-24 19:23:21.356481,0 days 00:00:00.016098,0.017538,balanced,COMPLETE
43,43,0.935,2026-06-24 19:23:21.372469,2026-06-24 19:23:21.387751,0 days 00:00:00.015282,0.020708,balanced,COMPLETE
68,68,0.935,2026-06-24 19:23:21.767927,2026-06-24 19:23:21.782949,0 days 00:00:00.015022,0.031580,balanced,COMPLETE


In [23]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])
    
    model = LogisticRegression(
        class_weight=class_weight,
        C=c,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='f1',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

log_reg_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_log_reg_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_log_reg_unscaled_f1_optuna_results)

[I 2026-06-24 19:23:22,294] A new study created in memory with name: no-name-0bab2b58-3f0f-41e4-b560-d8e548ad01dd
[I 2026-06-24 19:23:22,322] Trial 0 finished with value: 0.7231606147150822 and parameters: {'C': 0.7429561565982611, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.7231606147150822.
[I 2026-06-24 19:23:22,337] Trial 1 finished with value: 0.7637535014005602 and parameters: {'C': 0.4935644037518289, 'Class_weight': None}. Best is trial 1 with value: 0.7637535014005602.
[I 2026-06-24 19:23:22,353] Trial 2 finished with value: 0.7637535014005602 and parameters: {'C': 0.8941397917695677, 'Class_weight': None}. Best is trial 1 with value: 0.7637535014005602.
[I 2026-06-24 19:23:22,368] Trial 3 finished with value: 0.7266693866449068 and parameters: {'C': 0.41925089425399004, 'Class_weight': 'balanced'}. Best is trial 1 with value: 0.7637535014005602.
[I 2026-06-24 19:23:22,383] Trial 4 finished with value: 0.8063636363636363 and parameters: {'C': 0.11906713406829948

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,state
32,32,0.811667,2026-06-24 19:23:22.877184,2026-06-24 19:23:22.892282,0 days 00:00:00.015098,0.059693,None,COMPLETE
34,34,0.811667,2026-06-24 19:23:22.907750,2026-06-24 19:23:22.923709,0 days 00:00:00.015959,0.069756,None,COMPLETE
25,25,0.811667,2026-06-24 19:23:22.761329,2026-06-24 19:23:22.777192,0 days 00:00:00.015863,0.080574,None,COMPLETE
31,31,0.811667,2026-06-24 19:23:22.862041,2026-06-24 19:23:22.877184,0 days 00:00:00.015143,0.078626,None,COMPLETE
24,24,0.811667,2026-06-24 19:23:22.746410,2026-06-24 19:23:22.761329,0 days 00:00:00.014919,0.113611,None,COMPLETE


##### Tryout best params

In [24]:
log_reg_recall_optimized_model = LogisticRegression(C=0.769946, class_weight='balanced', random_state=random_state)
log_reg_recall_optimized_model.fit(X_train_scaled, y_train)
y_pred = log_reg_recall_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Logistic Regression (Recall Optimized)')

              precision    recall  f1-score   support

           0       0.98      0.92      0.95       190
           1       0.52      0.84      0.64        19

    accuracy                           0.91       209
   macro avg       0.75      0.88      0.80       209
weighted avg       0.94      0.91      0.92       209



In [25]:
log_reg_f1_optimized_model = LogisticRegression(C=0.315005, class_weight='balanced', random_state=random_state)
log_reg_f1_optimized_model.fit(X_train_scaled, y_train)
y_pred = log_reg_f1_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Logistic Regression (F1 Optimized)')

              precision    recall  f1-score   support

           0       0.98      0.92      0.95       190
           1       0.52      0.84      0.64        19

    accuracy                           0.91       209
   macro avg       0.75      0.88      0.80       209
weighted avg       0.94      0.91      0.92       209



#### Ridge

In [26]:
# We can tune alpha and class weight

##### Recall optimized

In [27]:
def objective(trial):
    alpha = trial.suggest_float("alpha", 0.001, 1.0)
    class_weight = trial.suggest_categorical("class_weight", ['balanced', None])
    
    model = RidgeClassifier(
        class_weight=class_weight,
        alpha=alpha,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

ridge_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_ridge_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_ridge_unscaled_recall_optuna_results)

[I 2026-06-24 19:23:24,169] A new study created in memory with name: no-name-f5599b59-5e48-4a69-ae09-f86d8cfea12f
[I 2026-06-24 19:23:24,190] Trial 0 finished with value: 0.9350000000000002 and parameters: {'alpha': 0.07830805371391322, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.9350000000000002.
[I 2026-06-24 19:23:24,206] Trial 1 finished with value: 0.7899999999999999 and parameters: {'alpha': 0.43514992988920276, 'class_weight': None}. Best is trial 0 with value: 0.9350000000000002.
[I 2026-06-24 19:23:24,222] Trial 2 finished with value: 0.9350000000000002 and parameters: {'alpha': 0.8585361598234713, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.9350000000000002.
[I 2026-06-24 19:23:24,237] Trial 3 finished with value: 0.9350000000000002 and parameters: {'alpha': 0.7703638839486203, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.9350000000000002.
[I 2026-06-24 19:23:24,253] Trial 4 finished with value: 0.7899999999999999 and parameters: {'

,number,value,datetime_start,datetime_complete,duration,params_alpha,params_class_weight,state
0,0,0.935,2026-06-24 19:23:24.170528,2026-06-24 19:23:24.190455,0 days 00:00:00.019927,0.078308,balanced,COMPLETE
2,2,0.935,2026-06-24 19:23:24.207075,2026-06-24 19:23:24.222367,0 days 00:00:00.015292,0.858536,balanced,COMPLETE
3,3,0.935,2026-06-24 19:23:24.222367,2026-06-24 19:23:24.237571,0 days 00:00:00.015204,0.770364,balanced,COMPLETE
6,6,0.935,2026-06-24 19:23:24.268651,2026-06-24 19:23:24.283779,0 days 00:00:00.015128,0.726871,balanced,COMPLETE
28,28,0.935,2026-06-24 19:23:24.622241,2026-06-24 19:23:24.637383,0 days 00:00:00.015142,0.934212,balanced,COMPLETE


##### Tryout best params

In [28]:
ridge_recall_optimized_model = RidgeClassifier(alpha=0.039558, class_weight='balanced', random_state=random_state)
ridge_recall_optimized_model.fit(X_train_scaled, y_train)
y_pred = ridge_recall_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Ridge Classifier (Recall Optimized)')

              precision    recall  f1-score   support

           0       0.98      0.88      0.93       190
           1       0.42      0.84      0.56        19

    accuracy                           0.88       209
   macro avg       0.70      0.86      0.75       209
weighted avg       0.93      0.88      0.90       209



##### F1 optimized

In [29]:
def objective(trial):
    alpha = trial.suggest_float("alpha", 0.001, 1.0)
    class_weight = trial.suggest_categorical("class_weight", ['balanced', None])
    
    model = RidgeClassifier(
        class_weight=class_weight,
        alpha=alpha,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='f1',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

ridge_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_ridge_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_ridge_unscaled_f1_optuna_results)

[I 2026-06-24 19:23:25,795] A new study created in memory with name: no-name-8bd87806-44f5-46bf-af8c-02c0baeabfea
[I 2026-06-24 19:23:25,823] Trial 0 finished with value: 0.6647694334650855 and parameters: {'alpha': 0.8233293275412075, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.6647694334650855.
[I 2026-06-24 19:23:25,845] Trial 1 finished with value: 0.7599343185550083 and parameters: {'alpha': 0.5555056826845673, 'class_weight': None}. Best is trial 1 with value: 0.7599343185550083.
[I 2026-06-24 19:23:25,860] Trial 2 finished with value: 0.7599343185550083 and parameters: {'alpha': 0.26924662542734534, 'class_weight': None}. Best is trial 1 with value: 0.7599343185550083.
[I 2026-06-24 19:23:25,876] Trial 3 finished with value: 0.6647694334650855 and parameters: {'alpha': 0.0033975815539298133, 'class_weight': 'balanced'}. Best is trial 1 with value: 0.7599343185550083.
[I 2026-06-24 19:23:25,894] Trial 4 finished with value: 0.6647694334650855 and parameters: {'alph

,number,value,datetime_start,datetime_complete,duration,params_alpha,params_class_weight,state
1,1,0.759934,2026-06-24 19:23:25.823427,2026-06-24 19:23:25.845475,0 days 00:00:00.022048,0.555506,None,COMPLETE
2,2,0.759934,2026-06-24 19:23:25.845475,2026-06-24 19:23:25.860983,0 days 00:00:00.015508,0.269247,None,COMPLETE
11,11,0.759934,2026-06-24 19:23:25.986723,2026-06-24 19:23:26.002082,0 days 00:00:00.015359,0.316546,None,COMPLETE
6,6,0.759934,2026-06-24 19:23:25.909788,2026-06-24 19:23:25.924701,0 days 00:00:00.014913,0.627226,None,COMPLETE
33,33,0.759934,2026-06-24 19:23:26.335403,2026-06-24 19:23:26.350559,0 days 00:00:00.015156,0.019088,None,COMPLETE


##### Tryout best params

In [30]:
ridge_f1_optimized_model = RidgeClassifier(alpha=0.364826, class_weight='balanced', random_state=random_state)
ridge_f1_optimized_model.fit(X_train_scaled, y_train)
y_pred = ridge_f1_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Ridge Classifier (F1 Optimized)')

              precision    recall  f1-score   support

           0       0.98      0.88      0.93       190
           1       0.41      0.84      0.55        19

    accuracy                           0.88       209
   macro avg       0.70      0.86      0.74       209
weighted avg       0.93      0.88      0.89       209



#### Lasso Logistic Regression

In [31]:
# We can only tune the C (float) and Class_weight to tryout balanced. 

In [32]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])
    
    model = LogisticRegression(
        penalty='l1', # Lasso
        class_weight=class_weight,
        C=c,
        solver='liblinear',
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15, n_jobs=-1)

lasso_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_lasso_unscaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_lasso_unscaled_recall_optuna_results)

[I 2026-06-24 19:23:27,489] A new study created in memory with name: no-name-fcac938c-0e98-4c66-8312-de74bfb833b4
[I 2026-06-24 19:23:27,591] Trial 0 finished with value: 0.9225000000000001 and parameters: {'C': 0.41346256526724884, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.9225000000000001.
[I 2026-06-24 19:23:27,606] Trial 5 finished with value: 0.9225000000000001 and parameters: {'C': 0.6387371254822808, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.9225000000000001.
[I 2026-06-24 19:23:27,609] Trial 1 finished with value: 0.8300000000000001 and parameters: {'C': 0.16434847869122923, 'Class_weight': None}. Best is trial 0 with value: 0.9225000000000001.
[I 2026-06-24 19:23:27,623] Trial 3 finished with value: 0.8166666666666668 and parameters: {'C': 0.629062958954625, 'Class_weight': None}. Best is trial 0 with value: 0.9225000000000001.
[I 2026-06-24 19:23:27,646] Trial 7 finished with value: 0.9350000000000002 and parameters: {'C': 0.1044316485223579,

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,state
14,14,0.9350,2026-06-24 19:23:27.528184,2026-06-24 19:23:27.648618,0 days 00:00:00.120434,0.259026,balanced,COMPLETE
9,9,0.9350,2026-06-24 19:23:27.514594,2026-06-24 19:23:27.648618,0 days 00:00:00.134024,0.242978,balanced,COMPLETE
7,7,0.9350,2026-06-24 19:23:27.508594,2026-06-24 19:23:27.646496,0 days 00:00:00.137902,0.104432,balanced,COMPLETE
12,12,0.9350,2026-06-24 19:23:27.522606,2026-06-24 19:23:27.647617,0 days 00:00:00.125011,0.264063,balanced,COMPLETE
0,0,0.9225,2026-06-24 19:23:27.491893,2026-06-24 19:23:27.591659,0 days 00:00:00.099766,0.413463,balanced,COMPLETE


In [33]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])
    
    model = LogisticRegression(
        penalty='l1', # Lasso
        class_weight=class_weight,
        C=c,
        solver='liblinear',
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='f1',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)

lasso_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_lasso_unscaled_f1_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_lasso_unscaled_f1_optuna_results)

[I 2026-06-24 19:23:27,683] A new study created in memory with name: no-name-d78a5173-e6f7-443d-8a37-2bb7a45f31d3
[I 2026-06-24 19:23:27,708] Trial 0 finished with value: 0.7222786696470908 and parameters: {'C': 0.8226769121681857, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.7222786696470908.
[I 2026-06-24 19:23:27,748] Trial 1 finished with value: 0.7998768472906403 and parameters: {'C': 0.3645916710328363, 'Class_weight': None}. Best is trial 1 with value: 0.7998768472906403.
[I 2026-06-24 19:23:27,779] Trial 2 finished with value: 0.7153717627401839 and parameters: {'C': 0.9362627395766104, 'Class_weight': 'balanced'}. Best is trial 1 with value: 0.7998768472906403.
[I 2026-06-24 19:23:27,795] Trial 3 finished with value: 0.6642818428184282 and parameters: {'C': 0.01650026900993542, 'Class_weight': 'balanced'}. Best is trial 1 with value: 0.7998768472906403.
[I 2026-06-24 19:23:27,811] Trial 4 finished with value: 0.7191144527986633 and parameters: {'C': 0.24461388194

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,state
5,5,0.805522,2026-06-24 19:23:27.812364,2026-06-24 19:23:27.826430,0 days 00:00:00.014066,0.202394,None,COMPLETE
14,14,0.805522,2026-06-24 19:23:28.001689,2026-06-24 19:23:28.015744,0 days 00:00:00.014055,0.116319,None,COMPLETE
13,13,0.805522,2026-06-24 19:23:27.977931,2026-06-24 19:23:28.000615,0 days 00:00:00.022684,0.153908,None,COMPLETE
12,12,0.805522,2026-06-24 19:23:27.951404,2026-06-24 19:23:27.977931,0 days 00:00:00.026527,0.184036,None,COMPLETE
11,11,0.799877,2026-06-24 19:23:27.932575,2026-06-24 19:23:27.951404,0 days 00:00:00.018829,0.279604,None,COMPLETE


##### Tryout best params

In [34]:
lasso_recall_optimized_model = LogisticRegression(
    penalty='l1', # Lasso
    C=0.248927, 
    class_weight=None,
    solver='saga',
    random_state=random_state
)
lasso_recall_optimized_model.fit(X_train_scaled, y_train)
y_pred = lasso_recall_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Lasso Classifier (Recall Optimized)')

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


              precision    recall  f1-score   support

           0       0.98      0.97      0.97       190
           1       0.71      0.79      0.75        19

    accuracy                           0.95       209
   macro avg       0.85      0.88      0.86       209
weighted avg       0.95      0.95      0.95       209



In [35]:
lasso_f1_optimized_model = LogisticRegression(
    penalty='l1', # Lasso
    C=0.496023, 
    class_weight=None,
    solver='saga',
    random_state=random_state
)
lasso_f1_optimized_model.fit(X_train_scaled, y_train)
y_pred = lasso_f1_optimized_model.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Lasso Classifier (F1 Optimized)')

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


              precision    recall  f1-score   support

           0       0.98      0.96      0.97       190
           1       0.68      0.79      0.73        19

    accuracy                           0.95       209
   macro avg       0.83      0.88      0.85       209
weighted avg       0.95      0.95      0.95       209



### Conclusions

In [36]:
reports_df = pd.concat(classification_reports, keys=classification_report_keys)
reports_df

precision    recall  \
Baseline Logistic Regression            0              0.988304  0.889474   
                                        1              0.447368  0.894737   
                                        accuracy       0.889952  0.889952   
                                        macro avg      0.717836  0.892105   
                                        weighted avg   0.939128  0.889952   
Baseline Ridge Classifier               0              0.978378  0.952632   
                                        1              0.625000  0.789474   
                                        accuracy       0.937799  0.937799   
                                        macro avg      0.801689  0.871053   
                                        weighted avg   0.946253  0.937799   
Baseline L1 (Lasso) Logistic Regression 0              0.973545  0.968421   
                                        1              0.700000  0.736842   
                                        accuracy       0.947368  0.947368   
                                        macro avg      0.836772  0.852632   
                                        weighted avg   0.948677  0.947368   
Logistic Regression (Recall Optimized)  0              0.983146  0.921053   
                                        1              0.516129  0.842105   
                                        accuracy       0.913876  0.913876   
                                        macro avg      0.749638  0.881579   
                                        weighted avg   0.940690  0.913876   
Logistic Regression (F1 Optimized)      0              0.983146  0.921053   
                                        1              0.516129  0.842105   
                                        accuracy       0.913876  0.913876   
                                        macro avg      0.749638  0.881579   
                                        weighted avg   0.940690  0.913876   
Ridge Classifier (Recall Optimized)     0              0.982456  0.884211   
                                        1              0.421053  0.842105   
                                        accuracy       0.880383  0.880383   
                                        macro avg      0.701754  0.863158   
                                        weighted avg   0.931419  0.880383   
Ridge Classifier (F1 Optimized)         0              0.982353  0.878947   
                                        1              0.410256  0.842105   
                                        accuracy       0.875598  0.875598   
                                        macro avg      0.696305  0.860526   
                                        weighted avg   0.930344  0.875598   
Lasso Classifier (Recall Optimized)     0              0.978723  0.968421   
                                        1              0.714286  0.789474   
                                        accuracy       0.952153  0.952153   
                                        macro avg      0.846505  0.878947   
                                        weighted avg   0.954684  0.952153   
Lasso Classifier (F1 Optimized)         0              0.978610  0.963158   
                                        1              0.681818  0.789474   
                                        accuracy       0.947368  0.947368   
                                        macro avg      0.830214  0.876316   
                                        weighted avg   0.951629  0.947368   

                                                      f1-score     support  
Baseline Logistic Regression            0             0.936288  190.000000  
                                        1             0.596491   19.000000  
                                        accuracy      0.889952    0.889952  
                                        macro avg     0.766390  209.000000  
                                        weighted avg  0.905397  209.000000  
Baseline Ridge Classifier               0        

Since this is a fraud dataset, the Lasso Recall optimized model could be the best, with F1 of 0.75 for positive class.  The lasso drives the unhelpful weights to zero. 

The log reg consistently has low precision for the positive class, likely due to the multicolinearity. There could be inconsistency due to this causing large swings in the model. 

Ridge might not performed as well since coefs can be completely shrunk to 0

In [41]:
import numpy as np

pd.set_option('display.max_rows', None)

coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Ridge_Coef': ridge_f1_optimized_model.coef_,
})

# Sort features by the absolute size of their coefficients
coef_df['Abs_Coefficient'] = np.abs(coef_df['Ridge_Coef'])
coef_df = coef_df.sort_values(by='Abs_Coefficient', ascending=False)
coef_df

,Feature,Ridge_Coef,Abs_Coefficient
39,ForgivenessDate_int,-0.636292,0.636292
38,DateApproved_int,0.336484,0.336484
0,ProcessingMethod,-0.318839,0.318839
21,OriginatingLenderState,-0.189621,0.189621
6,ServicingLenderState,0.178688,0.178688
27,ForgivenPercentage,0.155812,0.155812
29,PROCEED_Diff,-0.148566,0.148566
2,LoanStatus,-0.128188,0.128188
31,PAYROLL_PROCEED_pct,0.114106,0.114106
16,RENT_PROCEED,0.097229,0.097229


In [ ]:
len(coef_df[coef_df['Abs_Coefficient'] == 0])

1

We can see 1 features coefs get shrunk to 0 by the Ridge model. This shows these features did not add additional information to the model and can be safely dropped

In [43]:
coefficients = lasso_f1_optimized_model.coef_[0]

# 3. Create a DataFrame to map features to their coefficients
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': coefficients
})

# 4. VIEW RETAINED FEATURES (Coefficient is NOT 0)
selected_features = coef_df[coef_df['Coefficient'] != 0]
print("Features Selected by Lasso:")
print(selected_features)

# 5. VIEW REJECTED FEATURES (Coefficient is exactly 0)
dropped_features = coef_df[coef_df['Coefficient'] == 0]
print("\nFeatures Ignored by Lasso:")
print(dropped_features['Feature'].tolist())

Features Selected by Lasso:
                   Feature  Coefficient
0         ProcessingMethod    -0.519262
1            BorrowerState     0.218712
2               LoanStatus    -0.251691
3                     Term    -0.043586
7      RuralUrbanIndicator     0.184038
9             LMIIndicator    -0.062952
10  BusinessAgeDescription     0.200309
11            ProjectState     0.218712
12            JobsReported    -0.081325
14         PAYROLL_PROCEED    -0.016060
16            RENT_PROCEED     0.152955
20            BusinessType    -0.012146
21  OriginatingLenderState    -0.199626
22                 Veteran     0.149011
23               NonProfit     0.045029
25            ApprovalDiff    -0.055810
26       NotForgivenAmount    -0.041735
27      ForgivenPercentage    -0.629629
29            PROCEED_Diff    -0.435194
37         PROCEED_Per_Job     0.166441
38        DateApproved_int    -0.052837
39     ForgivenessDate_int    -0.952909
40      LoanStatusDate_int    -0.203905

Features Ig

In [44]:
len(dropped_features)

18

Lasso dropped 18 features